# Custom NISAR RSLC to GUNW Processing with ISCE3

The NISAR mission produces geocoded unwrapped interferograms (GUNW) for nearest-neighbor date pairs. 


## 1. Search for RSLCs with openSEPPO

openSEPPO allows us to search for RSLCs, spatially subset them, and download isce3-formatted RSLCs of the subsets. 

### 1.1 Define some search parameters
Use [ASF Vertex](https://search.asf.alaska.edu/#/?zoom=5.900&center=-143.815,63.929&resultsLoaded=true&granule=NISAR_L1_PR_RSLC_033_164_A_035_2005_QPDH_A_20261023T134231_20261023T134308_P05023_N_F_J_001&sciProducts=RSLC&polygon=POLYGON((-148.3535%2064.5512,-146.5495%2064.5512,-146.5495%2065.1426,-148.3535%2065.1426,-148.3535%2064.5512))) or the [NISAR Observation Plan](https://experience.arcgis.com/experience/6052a864cd01459393884a7f751558e3) to find the track and frame for your AOI.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
start_time_after = "2026-06-26"
start_time_before = "2030-06-26"
track, frame = 113, 79

### 1.2 Perform the RSLC search with openSEPPO

In [3]:
from openseppo.cli import nisar_search as ns

def seppo_search(argstring):
    args = ns.myargsparse("seppo_nisar_search " + argstring)
    recs = ns.search_earthaccess(args)
    return ns._sort_for_group(recs)


recs = seppo_search(f"--track {track} --frame {frame} --start_time_after {start_time_after} --start_time_before {start_time_before} --product RSLC")

urls = [r["url_https"] for r in recs]
urls

['https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_024_113_D_079_4005_DHDH_A_20260704T011332_20260704T011406_P05023_N_F_J_001/NISAR_L1_PR_RSLC_024_113_D_079_4005_DHDH_A_20260704T011332_20260704T011406_P05023_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_025_113_D_079_4005_DHDH_A_20260716T011331_20260716T011405_P05023_N_F_J_001/NISAR_L1_PR_RSLC_025_113_D_079_4005_DHDH_A_20260716T011331_20260716T011405_P05023_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_028_113_D_079_4005_DHDH_A_20260821T011329_20260821T011403_P05023_N_F_J_001/NISAR_L1_PR_RSLC_028_113_D_079_4005_DHDH_A_20260821T011329_20260821T011403_P05023_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_029_113_D_079_4005_DHDH_A_20260902T011329_20260902T011403_P05023_N_F_J_001/NISAR_L1_PR_RSLC_029_113_

## 2. Download subset RSLCs

### 2.1 (Optional) Use an interactive AOI selector to find a well-known-text (wkt) rectangle over your AOI

Alternatively, you can use [Vertex](https://search.asf.alaska.edu/#/), which also generates wkt rectangles and shows the bounds of the valid data. The bounding box on the plot generated below may include some nodata values around the edges, so Vertex provides a more accurate view. 

In [4]:
from pathlib import Path
import sys
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from util.wkt import rectangular_aoi_select

aoi_map = rectangular_aoi_select(recs[0]["_geom"])
print("Click the ■ button to select an area of interest\n") 
display(aoi_map["map"])

Click the ■ button to select an area of interest



Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

### 2.2 Define a subset AOI and a directory in which to store the downloaded subset RSLCs

In [5]:
from pathlib import Path

aoi_wkt = "POLYGON((-99.3253 19.3513,-99.1792 19.3513,-99.1792 19.4674,-99.3253 19.4674,-99.3253 19.3513))"

rslc_dir = Path.home() / "mexico_city_RSLCs"
rslc_dir.mkdir(exist_ok=True)

gunw_dir = Path.home() / "mexico_city_GUNWs"
gunw_dir.mkdir(exist_ok=True)

### 2.3 Subset and download the RSLCs

In [6]:
import shapely
from pathlib import Path
from openseppo.cli import nisar_rslc_convert


minx, miny, maxx, maxy = shapely.from_wkt(aoi_wkt).bounds
ulx, uly, lrx, lry = minx, maxy, maxx, miny

for url in urls:
    cmd = f'nisar_rslc_convert -i {url} -o {rslc_dir} --all_freq --projwin {ulx} {uly} {lrx} {lry} -v'
    nisar_rslc_convert._main(cmd)

---> Detected Earthdata HTTPS URL. Using Earthdata credentials.
Starting RSLC Subset Processing: 1 file(s).
Mode: projwin [-99.3253, 19.4674, -99.1792, 19.3513] (EPSG:4326)
Freq: A | All frequencies: True
    [t] earthaccess login (cached token, expires 2026-11-21): instant

--> Processing RSLC: NISAR_L1_PR_RSLC_024_113_D_079_4005_DHDH_A_20260704T011332_20260704T011406_P05023_N_F_J_001.h5
    [WARN] USGS elevation lookup unavailable -- falling back to max 1000 m / min 0 m for this run.
           A successful lookup selects different geolocationGrid height levels, so subsets written now will NOT match subsets of the same granules written when the service answers.
           Pass --max_height/--min_height to pin the geometry (the service is US-only, so it always fails outside the US).
    Terrain heights (default): window max 1000 m, min 0 m, polygon 500 m -- resolved once, reused for every granule in this run
    Terrain max height: 1000m
    Terrain max height: 1000m
    Terrain max h

<hr>

## 3. Glob the paths to the RSLCs and define a network of RSLC pairs to process as GUNWs

### 3.1 Glob the paths to the downloaded RSLCs

In [7]:
rslcs = list(rslc_dir.glob("*.h5"))
rslcs.sort()
rslcs

[PosixPath('/home/jovyan/mexico_city_RSLCs/NISAR_L1_PR_RSLC_024_113_D_079_4005_DHDH_A_20260704T011332_20260704T011406_P05023_N_F_J_001_subset_az6926-10841_rg30903-35226.h5'),
 PosixPath('/home/jovyan/mexico_city_RSLCs/NISAR_L1_PR_RSLC_025_113_D_079_4005_DHDH_A_20260716T011331_20260716T011405_P05023_N_F_J_001_subset_az7428-11343_rg30907-35231.h5'),
 PosixPath('/home/jovyan/mexico_city_RSLCs/NISAR_L1_PR_RSLC_028_113_D_079_4005_DHDH_A_20260821T011329_20260821T011403_P05023_N_F_J_001_subset_az8332-12347_rg30906-35229.h5'),
 PosixPath('/home/jovyan/mexico_city_RSLCs/NISAR_L1_PR_RSLC_029_113_D_079_4005_DHDH_A_20260902T011329_20260902T011403_P05023_N_F_J_001_subset_az8031-12046_rg30903-35226.h5'),
 PosixPath('/home/jovyan/mexico_city_RSLCs/NISAR_L1_PR_RSLC_030_113_D_079_4005_DHDH_A_20260914T011329_20260914T011403_P05023_N_F_J_001_subset_az7729-11744_rg30932-35098.h5'),
 PosixPath('/home/jovyan/mexico_city_RSLCs/NISAR_L1_PR_RSLC_031_113_D_079_4005_DHDH_A_20260926T011329_20260926T011403_P05023_

### 3.2 Define a network of interferometric RSLC pairs by index

In [8]:
idx = [[0,1], [0,2]] #, [1,2], [1,3], [2,3], [2,4], [3,4], [3,5], [4,5], [4,6], [5,6], [5,7], [6,7]]

<hr>

## 4. Download the Copernicus DEM for NISAR, subset to the extents of the RSLCs

In [9]:
from util.rslc_gunw import stage_dem, get_rslc_summary

dem_path = gunw_dir / f"DEM_nisar.tif"
rslc_summary = get_rslc_summary(rslcs[0])
if not dem_path.exists():
    stage_dem([rslc_summary["footprint"]], dem_path) 

[11/04 23:17:13] [INFO dem.py] Bounds: -99.41061325335514 19.264515634954954 -99.09083850664486 19.552038445045046
[11/04 23:17:13] [INFO nisar_dem.py] Selected NISAR VRT: EPSG4326
[11/04 23:17:13] [INFO nisar_dem.py] Creating /home/jovyan/mexico_city_GUNWs/DEM_nisar.tif.part.tif
[11/04 23:17:13] [INFO nisar_dem.py] Fetching remote tiles...
[11/04 23:17:13] [INFO nisar_dem.py] Running GDAL command:
[11/04 23:17:13] [INFO nisar_dem.py] gdalwarp /vsicurl/https://nisar.asf.earthdatacloud.nasa.gov/NISAR/DEM/v1.2/EPSG4326/EPSG4326.vrt /home/jovyan/mexico_city_GUNWs/DEM_nisar.tif.part.tif -of GTiff -ot Float32 -te -99.4106944444444451 19.2643055555555556 -99.0906944444444377 19.5520833333333321 -tr 0.000277777777777777778 0.000277777777777777778 -wo NUM_THREADS=4 -r nearest -wm 5000 -multi


<hr>

## 5. Use ISCE3 to process GUNWs from the subset RSLCs

In [10]:
%%time

import argparse
import time
from pathlib import Path
from pyproj import Transformer
import numpy as np
import shutil
import h5py, shapely, yamale
from ruamel.yaml import YAML
import nisar.workflows
from nisar.workflows.insar_runconfig import InsarRunConfig
from nisar.workflows.stage_dem import point2epsg

from pathlib import Path
import sys
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from util.rslc_gunw import write_custom_gunw_config, run_insar

# Gather the EPSG of the AOI
aoi_geometry = shapely.from_wkt(aoi_wkt)
epsg = point2epsg(aoi_geometry.centroid.x, aoi_geometry.centroid.y)

# Transform AOI bounds to output EPSG 
x0, y0, x1, y1 = Transformer.from_crs(4326, epsg, always_xy=True).transform_bounds(*aoi_geometry.bounds, densify_pts=21)

# Snap bounds to an 80 meter grid
posting = 80.0
geocode_bounds = tuple(float(v) for v in (np.floor(x0 / posting) * posting, np.floor(y0 / posting) * posting,
                                          np.ceil(x1 / posting) * posting, np.ceil(y1 / posting) * posting))
print("EPSG", epsg, "box", geocode_bounds)  

# Create a ruamel.yaml.comments.CommentedMap object from the default gunw config
default_gunw_config_path = Path("../config/nisar_gunw_solid_earth_cpu_20_05.yaml").resolve()
schema = yamale.make_schema(Path(nisar.workflows.__file__).parent / "schemas" / "insar.yaml", parser="ruamel")
yaml = YAML()
yaml.default_flow_style = False
gunw_config = yaml.load(default_gunw_config_path)



for i_ref, i_sec in idx:
    # Collect RSLC paths and metadata
    ref_rslc_path, sec_rslc_path = Path(rslcs[i_ref]).resolve(), Path(rslcs[i_sec]).resolve()
    ref, sec = get_rslc_summary(ref_rslc_path), get_rslc_summary(sec_rslc_path)

    # Generate output GUNW filename
    polarization = "HH"
    gunw_name = (f"NISAR_L2_CUSTOM_GUNW_{ref['cycle']}_{ref['track']}_{ref['pass_direction']}_{ref['frame']}_{sec['cycle']}_{ref['mode']}"
                 f"_{polarization}_{ref['start_string']}_{ref['end_string']}_{sec['start_string']}_{sec['end_string']}.h5")
    
    # Skip processing if GUNW already exists
    gunw_path = gunw_dir / gunw_name
    if gunw_path.exists():
        print(f"skip (already done): {gunw_name}")
        continue
    print(f"\n=== {gunw_name}")

    # Create working directories and define paths for GUNW processing
    work_path = Path.home() / f"{gunw_dir.name}_work" / f"gunw_work_{i_ref}_{i_sec}"
    work_path.mkdir(parents=True, exist_ok=True)
    out_dir = work_path / "output"
    scratch_dir = work_path / "scratch"
    qa_dir = work_path / "qa"
    for d in (out_dir, scratch_dir, qa_dir):
        d.mkdir(parents=True, exist_ok=True)
    gunw_work_path = out_dir / gunw_name
    log_path = scratch_dir / "insar.log"
    paths = {
        "reference_rslc_file": ref_rslc_path,
        "secondary_rslc_file": sec_rslc_path,
        "work_path": work_path,
        "gunw_work_path": gunw_work_path,
        "dem_file": dem_path,
        "out_dir": out_dir,
        "scratch_dir": scratch_dir,
        "qa_dir": qa_dir,
        "log_path": log_path,
    }

    # Write a custom ISCE3 config for processing the GUNW
    runconfig = write_custom_gunw_config(gunw_config, ref, sec, paths, epsg, geocode_bounds, polarization, schema)

    # Validate custom config
    InsarRunConfig(argparse.Namespace(run_config_path=str(runconfig), log_file=False))

    # Run ISCE3 on the custom config
    t0 = time.time()
    run_insar(runconfig, work_path, log_path)
    shutil.copy2(gunw_work_path, gunw_path)
    print(f"  done in {(time.time() - t0) / 60:.1f} min -> {gunw_path}")


# Check that every GUNW has the same grid
grids = {}
for f in sorted(gunw_dir.glob("*GUNW_*.h5")):
    with h5py.File(f, "r") as h:
        g = h["science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram"]
        x, y = g["xCoordinates"][()], g["yCoordinates"][()]
    grids[f.name] = (y.size, x.size, round(float(x[0]), 3), round(float(y[0]), 3))
    print(f"{f.name}  shape={y.size}x{x.size}  x0={x[0]:.1f}  y0={y[0]:.1f}")
if len(set(grids.values())) == 1:
    print("\nOK: all GUNWs share one grid")
else:
    print("\nWARNING: GUNWs are on different grids")

EPSG 32614 box (465760.0, 2139680.0, 481200.0, 2152640.0)

=== NISAR_L2_CUSTOM_GUNW_024_113_descending_79_025_DHDH_HH_20260704T011332_20260704T011406_20260716T011331_20260716T011405.h5


<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'


  [   0.1 min] successfully ran bandpass_insar in 0.029 seconds
  [   0.7 min] successfully ran rdr2geo in 35.130 seconds
  [   0.7 min] Successfully ran geo2rdr in 2.516 seconds
  [   0.8 min] successfully ran prepare_insar_hdf5 in 4.438 seconds
  [   0.8 min] successfully ran resample for frequency A in 4.618 seconds
  [   0.8 min] successfully ran resample in 4.618 seconds
  [   1.0 min] Successfully ran dense_offsets in 9.241 seconds
  [   1.1 min] Successfully ran polyfit rubbersheet in 3.975 seconds
  [   1.2 min] successfully ran resample for frequency A in 4.491 seconds
  [   1.2 min] successfully ran resample in 4.492 seconds
  [   1.3 min] successfully ran crossmul in 6.480 seconds
  [   1.3 min] successfully ran crossmul in 6.153 seconds
  [   1.4 min] Successfully ran phase unwrapping in 7.257 seconds
  [   1.4 min] successfully ran split_spectrum in 0.000 seconds
  [   1.5 min] successfully ran rdr2geo in 4.486 seconds
  [   1.5 min] Successfully ran geo2rdr in 0.368 secon

<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1: SyntaxWarning: invalid escape sequence '\d'


  [   0.1 min] successfully ran bandpass_insar in 0.029 seconds
  [   0.7 min] successfully ran rdr2geo in 35.047 seconds
  [   0.7 min] Successfully ran geo2rdr in 2.632 seconds
  [   0.8 min] successfully ran prepare_insar_hdf5 in 4.453 seconds
  [   0.8 min] successfully ran resample for frequency A in 4.405 seconds
  [   0.8 min] successfully ran resample in 4.406 seconds
  [   1.0 min] Successfully ran dense_offsets in 9.018 seconds
  [   1.1 min] Successfully ran polyfit rubbersheet in 3.879 seconds
  [   1.2 min] successfully ran resample for frequency A in 4.366 seconds
  [   1.2 min] successfully ran resample in 4.366 seconds
  [   1.3 min] successfully ran crossmul in 6.461 seconds
  [   1.3 min] successfully ran crossmul in 6.143 seconds
  [   1.3 min] Successfully ran phase unwrapping in 7.455 seconds
  [   1.4 min] successfully ran split_spectrum in 0.000 seconds
  [   1.4 min] successfully ran rdr2geo in 4.413 seconds
  [   1.5 min] Successfully ran geo2rdr in 0.369 secon